### k=15 Nearest Neighbors matching

To identify the most abundant cell type in each neighborhood, we performed k-nearest neighbor (kNN) classification using the NearestNeighbors function from scikit-learn. We employed the Ball Tree algorithm to fit the Stereo-seq spatial coordinates as a reference and queried the 15 nearest Stereo-seq neighbors for each qMSI spot. The most prevalent Stereo-seq cell type among the neighbors was assigned to each qMSI spot, excluding 'Undetermined' cell types.

In [21]:
import anndata as ad
import pandas as pd
import numpy as np
from sklearn.neighbors import NearestNeighbors
from collections import Counter
import matplotlib.pyplot as plt
import matplotlib.cm as cm

In [18]:
banksy_output = "/exports/humgen/bmanzato/banksy_data"
qmsi_data = "/exports/humgen/bmanzato/nieromics_dir/qMSI_data"
lts = "/exports/archive/hg-funcgenom-research/IRI_multimodal_project/Stereo-seq_IRI"
nieromics_dir = "/exports/humgen/bmanzato/nieromics_dir"

#### Metabolomics

In [40]:
# look up df
met_ann = pd.read_csv(f"{nieromics_dir}/spatial_multiomics_data_MS/metabolomics/annotated_iri.csv",index_col=1)

# qmsi df (stseq coord space)
qmsi_transformed_biri1 = pd.read_csv(f"{nieromics_dir}/spatial_multiomics_data_MS/overlay/qmsi_transformed_biri1.csv",index_col=0)
qmsi_transformed_biri2 = pd.read_csv(f"{nieromics_dir}/spatial_multiomics_data_MS/overlay/qmsi_transformed_biri2_0403.csv",index_col=0)
qmsi_transformed_biri3 = pd.read_csv(f"{nieromics_dir}/spatial_multiomics_data_MS/overlay/qmsi_transformed_biri3.csv",index_col=0)
qmsi_transformed_biri1['sample'] = 'IRI1'
qmsi_transformed_biri2['sample'] = 'IRI2'
qmsi_transformed_biri3['sample'] = 'IRI3'
qmsi_transformed =  pd.concat([qmsi_transformed_biri1,qmsi_transformed_biri2,qmsi_transformed_biri3],axis=0)

# merge 
qmsi_transformed = qmsi_transformed.merge(met_ann[['ban_idents', 'fig_idents']], left_index=True, right_index=True, how='left')
qmsi_transformed['ban_idents'] = qmsi_transformed['ban_idents'].astype('Int64')
qmsi_transformed.columns = ['x','y','sample','qMSI_SD','qMSI_CT']
qmsi_transformed.head()

,x,y,sample,qMSI_SD,qMSI_CT
X,,,,,
Spot 132146,8223.410621,24443.357297,IRI1,3,PT-S1/S2
Spot 132147,8259.238379,24458.344966,IRI1,3,PT-S1/S2
Spot 132148,8295.066137,24473.332635,IRI1,3,PT-S1/S2
Spot 132150,8366.721543,24503.307927,IRI1,9,vessel
Spot 132328,8200.623101,24387.191346,IRI1,2,PT-S1/S2


#### Transcriptomics

In [43]:
stseq_ann = pd.read_csv(f"{nieromics_dir}/spatial_multiomics_data_MS/transcriptomics/metadata_complete.csv",index_col=0)
stseq_ann = stseq_ann[stseq_ann['condition']=='IRI']
stseq_ann['banksy'] = stseq_ann['banksy'].astype('Int64')
stseq_ann.columns = ['x','y','condition','sample','SS_SD','SS_CT']
stseq_ann.head()

,x,y,condition,sample,SS_SD,SS_CT
131_363_4,9902,19908,IRI,IRI1,11,PT-S3
236_306_4,14102,17628,IRI,IRI1,5,undetermined
208_431_4,12982,22628,IRI,IRI1,17,glomeruli
192_289_4,12342,16948,IRI,IRI1,7,Injured tubule
265_384_4,15262,20748,IRI,IRI1,5,CNT


In [76]:
# Initialize dict
all_ct_counts_sd7_total = {}
all_ct_counts_sd9_total = {}
all_ct_counts_sd234_total = {}

for sample in ['IRI1', 'IRI2', 'IRI3']:

    print(f"\n--- Sample: {sample} ---")

    all_ct_counts_sd7 = {}
    all_ct_counts_sd9 = {}
    all_ct_counts_sd234 = {}

    # Subset
    qmsi_subset = qmsi_transformed[qmsi_transformed['sample'] == sample].copy()  
    biri_subset = stseq_ann[stseq_ann['sample'] == sample]

    biri_coords = biri_subset[['x', 'y']].values
    qmsi_coords = qmsi_subset[['x', 'y']].values

    # k-NN for nearest neighbors
    nbrs = NearestNeighbors(n_neighbors=15, algorithm='ball_tree').fit(biri_coords)
    distances, indices = nbrs.kneighbors(qmsi_coords)

    # Get closest barcode and annotations
    closest_barcodes = np.array(biri_subset.index)[indices] 
    closest_ct_annotations = biri_subset.iloc[indices.flatten()]['SS_CT'].values.reshape(indices.shape).tolist()

    # Compute most prevalent cell types
    most_prevalent_ct = [most_common(ct_list) for ct_list in closest_ct_annotations]

    qmsi_subset['closest_ss_id'] = closest_barcodes[:, 0] 
    qmsi_subset['closest_CT_ss'] = closest_ct_annotations
    qmsi_subset['most_prevalent_CT_ss'] = most_prevalent_ct

    # Subset by qMSI_SD
    dfs = {
        "SD234": qmsi_subset[qmsi_subset['qMSI_SD'].isin([2, 3, 4])], # healthy 
        "SD7": qmsi_subset[qmsi_subset['qMSI_SD'].isin([7])], # inj
        "SD9": qmsi_subset[qmsi_subset['qMSI_SD'].isin([9])] # inj
    }

    # Count cell types per SD category
    for sd_label, df in dfs.items():
        ct_counts = df["most_prevalent_CT_ss"].value_counts()

        print(f"\n--- {sd_label} ---")
        if sd_label == "SD234":
            target_dict = all_ct_counts_sd234
        elif sd_label == "SD7":
            target_dict = all_ct_counts_sd7
        else:
            target_dict = all_ct_counts_sd9

        # Remove 'undetermined' 
        ct_counts = ct_counts[ct_counts.index != 'undetermined']

        for ct, count in ct_counts.items():
            if ct in target_dict:
                target_dict[ct] += count
            else:
                target_dict[ct] = count

        total_ct_count = sum(target_dict.values())

        # %
        for ct, count in target_dict.items():
            percentage = (count / total_ct_count) * 100
            print(f"{ct}: {percentage:.2f}%")

        # update total counts 
        if sd_label == "SD234":
            for ct, count in target_dict.items():
                if ct in all_ct_counts_sd234_total:
                    all_ct_counts_sd234_total[ct] += count
                else:
                    all_ct_counts_sd234_total[ct] = count
        elif sd_label == "SD7":
            for ct, count in target_dict.items():
                if ct in all_ct_counts_sd7_total:
                    all_ct_counts_sd7_total[ct] += count
                else:
                    all_ct_counts_sd7_total[ct] = count
        elif sd_label == "SD9":
            for ct, count in target_dict.items():
                if ct in all_ct_counts_sd9_total:
                    all_ct_counts_sd9_total[ct] += count
                else:
                    all_ct_counts_sd9_total[ct] = count

# calculate total cell type counts for each SD category
total_ct_count_sd7 = sum(all_ct_counts_sd7_total.values())
total_ct_count_sd9 = sum(all_ct_counts_sd9_total.values())
total_ct_count_sd234 = sum(all_ct_counts_sd234_total.values())

print("\n--- Total Across All Samples ---")
for sd_label, target_dict, total_ct_count in [("SD7", all_ct_counts_sd7_total, total_ct_count_sd7),
                                             ("SD9", all_ct_counts_sd9_total, total_ct_count_sd9),
                                             ("SD234", all_ct_counts_sd234_total, total_ct_count_sd234)]:
    print(f"\n--- {sd_label} ---")

    target_dict = {ct: count for ct, count in target_dict.items() if ct != 'undetermined'}
    
    for ct, count in target_dict.items():
        percentage = (count / total_ct_count) * 100
        print(f"{ct}: {percentage:.2f}%")

print("\n" + "-" * 50 + "\n")



--- Sample: IRI1 ---

--- SD234 ---
PT-S1/S2: 43.66%
PT-S3: 16.44%
interstitial cells: 11.27%
TAL: 7.61%
CNT: 7.15%
FR-PT: 6.22%
DCT: 2.77%
collecting duct: 2.34%
glomeruli: 1.38%
Injured tubule: 1.17%

--- SD7 ---
PT-S1/S2: 36.58%
interstitial cells: 22.88%
FR-PT: 18.29%
CNT: 5.90%
Injured tubule: 4.53%
glomeruli: 3.48%
TAL: 2.53%
DCT: 2.37%
collecting duct: 2.27%
PT-S3: 1.16%

--- SD9 ---
interstitial cells: 37.37%
PT-S1/S2: 18.76%
FR-PT: 15.57%
Injured tubule: 6.94%
CNT: 6.27%
PT-S3: 4.82%
TAL: 3.95%
glomeruli: 3.33%
collecting duct: 2.31%
DCT: 0.68%

--- Sample: IRI2 ---

--- SD234 ---
PT-S1/S2: 32.20%
interstitial cells: 22.19%
PT-S3: 17.19%
TAL: 11.49%
FR-PT: 7.85%
CNT: 4.67%
DCT: 1.39%
Injured tubule: 1.13%
collecting duct: 1.03%
glomeruli: 0.87%

--- SD7 ---
PT-S1/S2: 24.15%
interstitial cells: 21.80%
FR-PT: 19.32%
TAL: 10.57%
PT-S3: 8.49%
CNT: 6.66%
Injured tubule: 4.05%
collecting duct: 2.22%
glomeruli: 1.04%
DCT: 1.04%
gaps: 0.65%

--- SD9 ---
interstitial cells: 43.16%
PT-